# Team 13 · Model selection for monthly passenger demand
Run all cells locally from within the repository. The D3 notebook supplies the observed route-month rows, features, dates and passenger counts. Two independent 24-month forecast windows precede a reserved 24-month final test window. All validation forecasts within a window use one cutoff and assume the directional routes are supplied. No missing route-month is interpreted as zero demand.

In [ ]:
from pathlib import Path
from IPython.utils.capture import capture_output

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.git').exists())
d3 = root / 'team-13/deliverable3/notebooks/02-demand-features.ipynb'
with capture_output():
    get_ipython().run_line_magic('run', f'"{d3}"')
plt.close('all')
assert months.index.equals(X.index) and y.index.equals(X.index)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

airports_in_route = ['ORG_AIRPRT', 'DES_AIRPRT']
calendar = ['departure_year', 'month_sin', 'month_cos']

def tabular_model(estimator):
    encoder = ColumnTransformer([
        ('airports', OneHotEncoder(handle_unknown='ignore'), airports_in_route),
        ('calendar', StandardScaler(), calendar)
    ])
    return make_pipeline(encoder, estimator)

def seasonal_reference(training, future):
    # Repeat the latest pre-cutoff observation for the same route and calendar month.
    # Sparse routes fall back to their historical median, then the training median.
    history = training.assign(season=training.month.dt.month).sort_values('month')
    seasonal_key = airports_in_route + ['season']
    last = history.drop_duplicates(seasonal_key, keep='last').set_index(seasonal_key).OD_PAX
    route_median = history.groupby(airports_in_route).OD_PAX.median()
    query = future.assign(season=future.month.dt.month)
    prediction = last.reindex(pd.MultiIndex.from_frame(query[seasonal_key])).to_numpy()
    fallback = route_median.reindex(pd.MultiIndex.from_frame(query[airports_in_route])).to_numpy()
    return np.where(np.isfinite(prediction), prediction,
                    np.where(np.isfinite(fallback), fallback, history.OD_PAX.median()))

def metrics(actual, predicted):
    error = np.asarray(predicted) - np.asarray(actual)
    assert np.isfinite(error).all()
    total = np.abs(actual).sum()
    return {'rows': len(error), 'MAE': np.abs(error).mean(),
            'RMSE': np.sqrt(np.mean(error**2)),
            'WAPE (%)': 100*np.abs(error).sum()/total if total else np.nan}


In [ ]:
period = months.dt.to_period('M')
horizon = 24
test_start = period.max() - horizon + 1
predictions = []
windows = []
for origin in [test_start - 2*horizon, test_start - horizon]:
    train = period < origin
    valid = (period >= origin) & (period < origin + horizon)
    if not train.any() or period[valid].nunique() != horizon:
        raise ValueError('Insufficient monthly coverage for the planned backtest.')
    assert period[train].max() < period[valid].min() and period[valid].max() < test_start
    windows.append(f'{origin} through {origin+horizon-1} ({int(train.sum())} training rows)')
    future = panel.loc[valid, ['month', *airports_in_route]]
    candidates = {'Seasonal reference': seasonal_reference(panel.loc[train], future)}
    for name, estimator in [
        ('Ridge', Ridge(alpha=1.0, solver='lsqr')),
        ('Gradient boosting', GradientBoostingRegressor(n_estimators=100, max_depth=2,
                                  learning_rate=.05, loss='huber', random_state=42))
    ]:
        model = tabular_model(estimator).fit(X.loc[train], y.loc[train])
        candidates[name] = model.predict(X.loc[valid])
    for name, values in candidates.items():
        predictions.append(pd.DataFrame({'window': str(origin), 'candidate': name,
            'actual': y.loc[valid], 'forecast': np.maximum(0, values)}))
results = pd.concat(predictions)


In [ ]:
def summarize(columns):
    rows = []
    for key, group in results.groupby(columns):
        labels = key if isinstance(key, tuple) else (key,)
        rows.append({**dict(zip(columns, labels)), **metrics(group.actual, group.forecast)})
    return pd.DataFrame(rows)

pooled = summarize(['candidate']).sort_values(['MAE','RMSE','candidate'])
by_window = summarize(['window','candidate']).sort_values(['window','MAE'])
assert pooled.rows.nunique() == 1 and by_window.groupby('window').rows.nunique().eq(1).all()
winner = pooled.iloc[0]
runner_up = pooled.iloc[1]
tradeoffs = {
    'Seasonal reference': 'Transparent and cheap to refresh. Sparse routes often rely on a median fallback.',
    'Ridge': 'Shares additive airport and calendar effects across sparse markets. Linear year extrapolation and missing interactions can bias forecasts.',
    'Gradient boosting': 'Learns nonlinear seasonal and airport effects. Rare airports have little support and trees cannot extrapolate a year trend.'
}
private = root / 'data/interim/team-13-d4'
private.mkdir(parents=True, exist_ok=True)
def markdown_table(frame):
    lines = ['| ' + ' | '.join(frame.columns) + ' |', '|' + '---|' * len(frame.columns)]
    for row in frame.itertuples(index=False, name=None):
        lines.append('| ' + ' | '.join(f'{v:.3f}' if isinstance(v, float) else str(v) for v in row) + ' |')
    return '\n'.join(lines)

comparison = ['# Model comparison', *windows,
    f'Final test reserved from {test_start}. Each observed route-month has equal weight.',
    '## Pooled validation', markdown_table(pooled), '## By forecast origin', markdown_table(by_window),
    *[f'**{k}** — {v}' for k, v in tradeoffs.items()]]
(private / 'model-comparison.md').write_text('\n\n'.join(comparison)+'\n')
decision = (f"# D5 baseline decision\n\nCarry forward **{winner.candidate}**. "
            f"Pooled validation MAE was {winner.MAE:.2f} passengers, "
            f"{runner_up.MAE-winner.MAE:.2f} below the next candidate ({runner_up.candidate}). "
            f"RMSE was {winner.RMSE:.2f}. {tradeoffs[winner.candidate]} "
            f"Test the frozen specification from {test_start} in D5. Results apply only to observed markets.\n")
(private / 'baseline-decision.md').write_text(decision)
print(decision)
